# Laboratorio CNNs-XAI

Este cuaderno deja documentado el flujo completo del laboratorio: exploracion del dataset, preprocesamiento, construccion de la CNN, revision de hiperparametros, interpretabilidad visual y despliegue.

## Como leer este notebook

- Esta organizado en el mismo orden de los ejercicios del PDF.
- Prioriza claridad metodologica sobre ejecucion pesada dentro del cuaderno.
- Las corridas largas ya quedaron respaldadas en `outputs/` y se consultan aqui para mantener el notebook reproducible y ordenado.
- Cuando una celda puede tardar mucho, se deja un interruptor para usar artefactos ya generados.

## Idea central del proyecto

El valor de este laboratorio no es solo entrenar una CNN, sino mostrar por que la calidad del dataset, la forma de particionar y la interpretabilidad visual cambian la lectura real del modelo. Por eso este cuaderno distingue entre:

1. el checkpoint legado que se conserva para la demo publica, y
2. la lectura metodologicamente mas honesta despues de la auditoria de duplicados y fuga entre splits.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage
from PIL import Image as PILImage

from cnn_xai_lab.config import DATA_DIR, OUTPUTS_DIR, MODELS_DIR, IMAGE_SIZE, BATCH_SIZE, SEED
from cnn_xai_lab.data import (
    discover_images,
    enrich_with_compact_hash,
    deduplicate_dataframe,
    summarize_dataset,
    save_json,
    save_sample_mosaic,
    split_dataframe,
    summarize_splits,
    make_tf_dataset,
)
from cnn_xai_lab.modeling import build_cnn_model, find_last_conv_layer_name
from cnn_xai_lab.training import ExperimentConfig, run_experiments
from cnn_xai_lab.xai import (
    prepare_image,
    predict_probabilities,
    compute_saliency_map,
    make_gradcam_heatmap,
    save_interpretability_panel,
)
from tensorflow import keras

REPO_URL = "https://github.com/loreprint/lab_cnn_deep"
PUBLIC_APP_URL = "https://labcnndeep-loreuec.streamlit.app/"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## Ejercicio 1. Descarga y exploracion del dataset

En esta seccion se responden tres preguntas basicas:

- cuantas imagenes existen realmente,
- cuantas clases hay,
- y por que no basta con contar archivos sin auditar duplicados.

La decision metodologica importante aqui es calcular una huella visual compacta (`compact_hash`) antes de sacar conclusiones sobre el volumen real de datos.


In [ ]:
dataset_frame = discover_images(DATA_DIR)
hashed_frame = enrich_with_compact_hash(dataset_frame)
unique_frame = deduplicate_dataframe(hashed_frame)

raw_summary = summarize_dataset(hashed_frame)
unique_summary = summarize_dataset(unique_frame)

save_json(raw_summary, OUTPUTS_DIR / "dataset_summary_raw.json")
save_json(unique_summary, OUTPUTS_DIR / "dataset_summary.json")
save_sample_mosaic(dataset_frame, OUTPUTS_DIR / "dataset_mosaic.png")

summary_table = pd.DataFrame(
    [
        {
            "vista": "dataset_crudo",
            "num_imagenes": raw_summary["num_images"],
            "female": raw_summary["classes"].get("female", 0),
            "male": raw_summary["classes"].get("male", 0),
            "duplicados_detectados": raw_summary.get("duplicate_rows", 0),
        },
        {
            "vista": "dataset_unico",
            "num_imagenes": unique_summary["num_images"],
            "female": unique_summary["classes"].get("female", 0),
            "male": unique_summary["classes"].get("male", 0),
            "duplicados_detectados": 0,
        },
    ]
)
summary_table


In [ ]:
size_table = pd.DataFrame(
    {
        "metrica": ["width_min", "width_max", "width_mean", "height_min", "height_max", "height_mean"],
        "valor": [
            raw_summary["width"]["min"],
            raw_summary["width"]["max"],
            round(raw_summary["width"]["mean"], 2),
            raw_summary["height"]["min"],
            raw_summary["height"]["max"],
            round(raw_summary["height"]["mean"], 2),
        ],
    }
)
size_table


In [ ]:
mosaic_path = OUTPUTS_DIR / "dataset_mosaic.png"
if mosaic_path.exists():
    display(IPImage(filename=str(mosaic_path)))
else:
    print(f"No se encontro el mosaico en: {mosaic_path}")


### Lectura de la exploracion

Los hallazgos mas importantes fueron estos:

- el dataset crudo contiene `5418` imagenes;
- las clases estan relativamente balanceadas en bruto;
- la variabilidad de tamano es alta, por lo que el `resize` es obligatorio;
- tras deduplicar por huella visual, solo `1672` grupos quedan como ejemplos realmente unicos.

Este punto cambia toda la interpretacion del laboratorio: si se parte un dataset con demasiados duplicados sin controlarlos, la evaluacion puede quedar inflada por fuga entre `train`, `val` y `test`.


## Ejercicio 2. Preprocesamiento y particion

El flujo base de preprocesamiento es:

`lectura RGB -> resize 224x224 -> normalizacion [0,1] -> tensor -> particion`

### Justificacion

- Mantener el mismo tamano es indispensable porque la CNN espera tensores con forma fija.
- Mantener el color en RGB evita introducir inconsistencias artificiales entre imagenes.
- Partir despues de deduplicar por grupos visuales evita fuga metodologica entre entrenamiento, validacion y prueba.


In [ ]:
splits = split_dataframe(
    hashed_frame,
    seed=SEED,
    group_column="compact_hash",
    keep_group_duplicates_for_train=True,
)
split_summary = summarize_splits(splits)
save_json(split_summary, OUTPUTS_DIR / "split_summary.json")

split_rows = []
for split_name, split_info in split_summary.items():
    split_rows.append(
        {
            "split": split_name,
            "count": split_info["count"],
            "female": split_info["classes"].get("female", 0),
            "male": split_info["classes"].get("male", 0),
        }
    )

pd.DataFrame(split_rows)


In [ ]:
train_dataset = make_tf_dataset(splits["train"], IMAGE_SIZE, BATCH_SIZE, training=True, seed=SEED)
val_dataset = make_tf_dataset(splits["val"], IMAGE_SIZE, BATCH_SIZE, training=False, seed=SEED)
test_dataset = make_tf_dataset(splits["test"], IMAGE_SIZE, BATCH_SIZE, training=False, seed=SEED)

batch_images, batch_labels = next(iter(train_dataset.take(1)))
print("Batch de imagenes:", batch_images.shape)
print("Batch de etiquetas:", batch_labels.shape)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for axis, image, label in zip(axes, batch_images[:4], batch_labels[:4]):
    axis.imshow(image.numpy())
    axis.set_title(f"label={int(label.numpy())}")
    axis.axis("off")
fig.suptitle("Vista rapida del lote de entrenamiento", fontsize=14)
fig.tight_layout()
plt.show()


### Lectura del preprocesamiento

El split corregido que usamos en el proyecto deja una idea importante:

- `1672` corresponde a grupos unicos despues de deduplicacion,
- pero `train` puede reexpandir duplicados solo dentro de grupos autorizados para entrenamiento,
- sin mezclar informacion con `val` ni `test`.

Eso permite aprovechar mejor el volumen de datos sin romper la honestidad del protocolo de evaluacion.


## Ejercicio 3. Construccion y entrenamiento del modelo CNN

La CNN del proyecto usa una estructura clasica pero robusta:

- bloques `Conv2D + BatchNormalization + MaxPooling2D`,
- `SpatialDropout2D` en capas intermedias,
- `GlobalAveragePooling2D` para reducir parametros,
- una capa densa final con `Dropout`,
- salida sigmoide para clasificacion binaria.

Ademas, la version actual incorpora `L2 regularization` y `label smoothing`, porque el objetivo no es solo acertar en entrenamiento sino reducir sobreconfianza y mejorar estabilidad.


In [ ]:
demo_model = build_cnn_model(
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    base_filters=48,
    kernel_size=5,
    dense_units=128,
    dropout=0.40,
    learning_rate=5e-4,
    l2_strength=1e-4,
    label_smoothing=0.03,
)

demo_model.summary()


### Sobre el entrenamiento dentro de este notebook

Entrenar todas las corridas largas desde el cuaderno no es practico para un entregable limpio. Por eso se usa un interruptor:

- si `RUN_LONG_TRAINING = True`, el notebook puede lanzar una corrida compacta;
- si `False`, carga los artefactos ya generados en `outputs/experiments/`.

Esto mantiene el cuaderno reproducible, pero evita que la entrega dependa de horas de ejecucion cada vez que se abre.


In [ ]:
RUN_LONG_TRAINING = False

experiments = [
    ExperimentConfig(
        name="baseline_notebook",
        base_filters=24,
        kernel_size=3,
        dense_units=96,
        dropout=0.45,
        learning_rate=3.5e-4,
        l2_strength=1e-4,
        label_smoothing=0.03,
    ),
    ExperimentConfig(
        name="compact_notebook",
        base_filters=32,
        kernel_size=5,
        dense_units=96,
        dropout=0.50,
        learning_rate=5e-4,
        l2_strength=1e-4,
        label_smoothing=0.03,
    ),
]

if RUN_LONG_TRAINING:
    comparison, best_info = run_experiments(
        experiments=experiments,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        output_dir=OUTPUTS_DIR / "experiments_notebook",
        best_model_path=MODELS_DIR / "model_notebook.keras",
        image_size=IMAGE_SIZE,
        epochs=6,
    )
else:
    comparison = pd.read_csv(OUTPUTS_DIR / "experiments" / "comparison.csv")
    with (OUTPUTS_DIR / "experiments" / "best_experiment.json").open("r", encoding="utf-8") as handle:
        best_info = json.load(handle)

comparison


In [ ]:
best_info


In [ ]:
active_curves = OUTPUTS_DIR / "experiments" / best_info["best_experiment"] / "training_curves.png"
corrected_curves = OUTPUTS_DIR / "experiments" / "compact_k5_dropout" / "training_curves.png"

for label, path in [
    ("Curvas del checkpoint activo en la demo", active_curves),
    ("Curvas del experimento corregido mas importante", corrected_curves),
]:
    print(label)
    if path.exists():
        display(IPImage(filename=str(path)))
    else:
        print(f"No se encontro la imagen: {path}")


### Lectura de entrenamiento

Este proyecto necesita una interpretacion cuidadosa:

- `wider_k5_dropout` se conserva como checkpoint activo para la demo publica;
- `compact_k5_dropout` es la referencia metodologica mas importante despues de corregir la fuga;
- que el checkpoint legado tenga metricas altas no significa automaticamente que generalice bien en fotos externas.

En otras palabras: el entrenamiento sirve para mostrar tanto un resultado operativo como una leccion metodologica sobre por que los datos importan tanto como la arquitectura.


## Ejercicio 4. Ajuste de hiperparametros

La comparacion de hiperparametros no se usa solo para elegir "el mas alto". Se usa para responder dos preguntas distintas:

1. cual checkpoint conviene para una demo publica estable,
2. y cual configuracion resume mejor el comportamiento del pipeline corregido.


In [ ]:
display_columns = [
    "name",
    "base_filters",
    "kernel_size",
    "dense_units",
    "dropout",
    "learning_rate",
    "l2_strength",
    "label_smoothing",
    "test_accuracy",
    "test_auc",
    "test_loss",
]
comparison[display_columns]


### Lectura de hiperparametros

- `wider_k5_dropout` mantiene el mejor puntaje legado y por eso sigue siendo la base de la demo.
- `compact_k5_dropout` es el resultado que mejor representa la version auditada y corregida del laboratorio.
- `face_crop_k5` demuestra que centrar el rostro ayuda conceptualmente, pero todavia no supera al mejor resultado corregido.

La conclusion honesta es que la app puede verse convincente en ciertos ejemplos y aun asi fallar bastante en imagenes externas con estilo distinto al dataset.


## Ejercicio 5. Interpretabilidad visual (Saliency Map y Grad-CAM)

### Diferencia conceptual

- `Saliency Map` trabaja a nivel de pixeles sensibles: muestra que pequenos cambios en la entrada afectan mas la salida.
- `Grad-CAM` trabaja a nivel de regiones semanticas: usa una capa convolucional profunda para destacar zonas completas relevantes.

En la practica:

- `Saliency Map` suele ser mas fino y mas ruidoso.
- `Grad-CAM` suele ser mas facil de interpretar sobre el rostro.


In [ ]:
model = keras.models.load_model(MODELS_DIR / "model.keras")
last_conv_layer = find_last_conv_layer_name(model)

sample_row = None
prediction = None
image_batch = None

for _, row in splits["test"].head(40).iterrows():
    with PILImage.open(row["path"]) as image:
        candidate_batch = prepare_image(image, image_size=IMAGE_SIZE)
    candidate_prediction = predict_probabilities(model, candidate_batch)
    if candidate_prediction["predicted_label"] == row["class_name"]:
        sample_row = row
        prediction = candidate_prediction
        image_batch = candidate_batch
        break

if sample_row is None:
    raise RuntimeError("No se encontro una imagen correctamente clasificada en la muestra revisada.")

saliency = compute_saliency_map(model, image_batch, prediction["predicted_index"])
gradcam = make_gradcam_heatmap(model, image_batch, prediction["predicted_index"], layer_name=last_conv_layer)

panel_path = OUTPUTS_DIR / "interpretability" / "notebook_panel.png"
save_interpretability_panel(
    image_batch=image_batch,
    saliency=saliency,
    gradcam=gradcam,
    output_path=panel_path,
    title="Interpretabilidad desde notebook",
)

{
    "sample_path": sample_row["path"],
    "true_label": sample_row["class_name"],
    "predicted_label": prediction["predicted_label"],
    "female_probability": round(prediction["female_probability"], 4),
    "male_probability": round(prediction["male_probability"], 4),
    "last_conv_layer": last_conv_layer,
}


In [ ]:
if panel_path.exists():
    display(IPImage(filename=str(panel_path)))
else:
    print(f"No se encontro la visualizacion: {panel_path}")


### Lectura de interpretabilidad

Los mapas del proyecto suelen destacar principalmente:

- ojos,
- cejas,
- contorno facial,
- nariz,
- boca,
- linea del cabello.

Cuando aparecen activaciones fuertes en cuello, ropa o fondo cercano, eso sugiere dependencia parcial de contexto espurio. Esta parte fue decisiva para entender por que el modelo puede acertar dentro del dataset y aun asi equivocarse con fotos externas.


## Ejercicio 6. Despliegue con Streamlit

La aplicacion publica tiene dos capas:

1. una pestana de informe que explica el laboratorio, y
2. una demo interactiva que ejecuta inferencia y muestra probabilidades, `Saliency Map` y `Grad-CAM`.

### Flujo funcional

`imagen subida -> preprocesamiento -> prediccion -> probabilidades -> mapas XAI`

### Interpretacion correcta del despliegue

El despliegue demuestra integracion tecnica entre codigo, modelo y visualizaciones. No demuestra por si solo que el clasificador ya sea confiable para cualquier retrato externo.


In [ ]:
print("Repositorio:", REPO_URL)
print("App publica:", PUBLIC_APP_URL)
print()
print("Para ejecutar la app localmente desde la raiz del proyecto:")
print("streamlit run streamlit_app.py")


## Ejercicio 7. Presentacion y reflexion final

### Que conviene mostrar en clase

- una imagen donde el modelo acierte,
- una imagen donde el modelo falle,
- la diferencia entre ver la escena completa y ver una region mas facial,
- y la lectura de los mapas XAI para justificar la decision.

### Reflexion final

La conclusion mas importante no es que la CNN "aprendio bien" en abstracto, sino que:

- auditar el dataset cambio la lectura real del rendimiento,
- la interpretabilidad ayudo a detectar dependencia de contexto,
- y la app sirve mejor como demostracion critica del pipeline que como clasificador plenamente confiable.

Esa conclusion hace que el cuaderno, el manuscrito y la pagina queden alineados en un mismo mensaje: el proyecto esta bien construido, pero tambien esta bien interpretado.


## Entrega final

Este notebook acompana los demas componentes del laboratorio:

- repositorio del proyecto,
- app publica en Streamlit,
- entregable manuscrito,
- artefactos en `outputs/`,
- y documentacion en `docs/`.

### Recomendacion final

Antes de entregar, completa en los documentos:

- nombre del estudiante,
- nombre del curso,
- enlace publico final,
- y una conclusion breve en primera persona sobre lo aprendido.
